# ChromaDB Chunk Explorer

Research notebook for inspecting, querying, and diagnosing the `knowledge_global` chunk collection.

Run from the repo root so relative paths resolve correctly:
```
jupyter notebook notebooks/chromadb_explorer.ipynb
```

In [ ]:
import sys, os
# Add repo root to path so packages resolve
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import re
import chromadb
from collections import Counter
import pandas as pd

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 50)

## 1. Connect and overview

In [ ]:
CHROMA_PATH = '../data/chroma'
COLLECTION  = 'knowledge_global'

client = chromadb.PersistentClient(path=CHROMA_PATH)
col    = client.get_collection(COLLECTION)

print(f'Collection : {COLLECTION}')
print(f'Total chunks: {col.count()}')
print(f'All collections: {[c.name for c in client.list_collections()]}')

In [ ]:
# Load everything into a DataFrame for easy slicing
raw = col.get(include=['metadatas', 'documents', 'embeddings'])

df = pd.DataFrame(raw['metadatas'])
df['document']    = raw['documents']
df['_id']         = raw['ids']
df['_emb_norm']   = [sum(x**2 for x in e)**0.5 for e in raw['embeddings']]  # sanity: all should be ~1.0 for nomic
df['_ot_len']     = df['original_text'].str.len()
df['_doc_len']    = df['document'].str.len()

print(df.shape)
df.dtypes

In [ ]:
# Collection-level stats
display(df[['source_type','access_level','scope']].apply(lambda c: c.value_counts()).fillna(0).astype(int))
print('\ntopic (top 25):')
display(df['topic'].value_counts().head(25))

## 2. Length distribution

In [ ]:
bins = [0, 100, 300, 1000, 3000, 10000, float('inf')]
labels = ['<100', '100-300', '300-1k', '1k-3k', '3k-10k', '>10k']
df['_len_bucket'] = pd.cut(df['_ot_len'], bins=bins, labels=labels)
display(df['_len_bucket'].value_counts().sort_index().rename('chunk count'))
print(f"\navg: {df['_ot_len'].mean():.0f}  median: {df['_ot_len'].median():.0f}  max: {df['_ot_len'].max()}  min: {df['_ot_len'].min()}")

In [ ]:
try:
    import matplotlib.pyplot as plt
    df['_ot_len'].clip(upper=5000).hist(bins=60, figsize=(12, 4))
    plt.xlabel('original_text length (chars, clipped at 5000)')
    plt.ylabel('chunk count')
    plt.title('Chunk length distribution')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib not installed — skipping plot')

## 3. Inspect individual chunks

In [ ]:
def show_chunk(row_or_index, chars=1000):
    """Pretty-print a chunk. Pass a DataFrame row or integer chunk_index."""
    if isinstance(row_or_index, int):
        rows = df[df['chunk_index'] == row_or_index]
        if rows.empty:
            print(f'chunk_index {row_or_index} not found')
            return
        row = rows.iloc[0]
    else:
        row = row_or_index

    ot = row.get('original_text', '')
    print(f"chunk_index : {row.get('chunk_index')}")
    print(f"breadcrumb  : {row.get('breadcrumb')}")
    print(f"headline    : {row.get('headline')}")
    print(f"summary     : {row.get('summary')}")
    print(f"topic       : {row.get('topic')}")
    print(f"access_level: {row.get('access_level')}")
    print(f"len(ot)     : {len(ot)}")
    print()
    print('--- original_text ---')
    print(ot[:chars])
    if len(ot) > chars:
        print(f'... [{len(ot) - chars} more chars]')

In [ ]:
# Example: inspect by chunk_index
show_chunk(14)

In [ ]:
# Browse a range of chunks
START, END = 90, 100
for _, row in df[df['chunk_index'].between(START, END)].iterrows():
    show_chunk(row, chars=300)
    print('=' * 60)

## 4. Problem detection

In [ ]:
# Stub chunks — very short original_text
stubs = df[df['_ot_len'] < 150].sort_values('_ot_len')
print(f'Stub chunks (<150 chars): {len(stubs)}')
display(stubs[['chunk_index','_ot_len','breadcrumb','headline','original_text']].head(20))

In [ ]:
# Giant chunks — dangerously large
giants = df[df['_ot_len'] > 5000].sort_values('_ot_len', ascending=False)
print(f'Giant chunks (>5000 chars): {len(giants)}')
display(giants[['chunk_index','_ot_len','breadcrumb','headline','topic']])

In [ ]:
# Encoding artifacts — unicode replacement chars
_artifact = re.compile(r'\ufffd|\u0092|\u0093|\u0094|\u0096|\u0097')
df['_has_artifact'] = df['original_text'].str.contains(_artifact)
artifact_chunks = df[df['_has_artifact']]
print(f'Chunks with encoding artifacts: {len(artifact_chunks)} / {len(df)}')
display(artifact_chunks[['chunk_index','breadcrumb','original_text']].head(10))

In [ ]:
# Mid-word line-wrap breaks (Word-\nbreak pattern)
_hyphen_break = re.compile(r'\w+- *\n *\w')
df['_has_linebreak'] = df['original_text'].apply(lambda t: bool(_hyphen_break.search(t)))
broken = df[df['_has_linebreak']]
print(f'Mid-word hyphen-break chunks: {len(broken)}')
display(broken[['chunk_index','breadcrumb','original_text']].head(10))

In [ ]:
# Image placeholder markup
_img = re.compile(r'==> picture|Start of picture text|End of picture text')
img_chunks = df[df['original_text'].str.contains(_img)]
print(f'Chunks with image placeholder markup: {len(img_chunks)}')
display(img_chunks[['chunk_index','breadcrumb','headline']].head(15))

In [ ]:
# TOC noise — lots of dot-leaders
_toc = re.compile(r'\.{5,}|\-{5,}')
toc_chunks = df[df['original_text'].str.contains(_toc)]
print(f'TOC/noise chunks: {len(toc_chunks)}')
display(toc_chunks[['chunk_index','_ot_len','breadcrumb','headline']].head(10))

In [ ]:
# Garbled OCR — high ratio of very short lines
def _garbled_ratio(text):
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if not lines:
        return 0.0
    return sum(1 for l in lines if len(l) <= 3) / len(lines)

df['_garbled'] = df['original_text'].apply(_garbled_ratio)
garbled = df[df['_garbled'] > 0.35].sort_values('_garbled', ascending=False)
print(f'Likely garbled/OCR-noise chunks: {len(garbled)}')
display(garbled[['chunk_index','_garbled','_ot_len','breadcrumb','headline']].head(15))

In [ ]:
# Stranded page numbers — bare integer lines
_page_num = re.compile(r'(?m)^\s*\d{1,4}\s*$')
df['_has_pagenum'] = df['original_text'].apply(lambda t: bool(_page_num.search(t)))
paged = df[df['_has_pagenum']]
print(f'Chunks with stranded page numbers: {len(paged)}')
display(paged[['chunk_index','breadcrumb']].head(15))

In [ ]:
# Breadcrumb collision — multiple chunks sharing the same breadcrumb
bc_counts = df.groupby('breadcrumb').size().reset_index(name='count').sort_values('count', ascending=False)
print('Breadcrumbs shared by >1 chunk (top 20):')
display(bc_counts[bc_counts['count'] > 1].head(20))

In [ ]:
# Broken breadcrumbs — bold/italic noise, mid-sentence, or single letter
_bold_bc  = df['breadcrumb'].str.contains(r'\*\*|_\*|\*_', regex=True)
_short_bc = df['breadcrumb'].str.len() < 25
bad_bc = df[_bold_bc | _short_bc]
print(f'Breadcrumbs with bold/italic markers or very short: {len(bad_bc)}')
display(bad_bc[['chunk_index','breadcrumb']].drop_duplicates('breadcrumb').head(25))

## 5. Semantic search against ChromaDB

In [ ]:
# Embed a query and retrieve top-k — requires Ollama running locally
try:
    import sys
    sys.path.insert(0, os.path.join(ROOT, 'packages/rag'))
    from rag.knowledge.embedder import OllamaEmbedFn
    from core.config import settings

    embed_fn = OllamaEmbedFn(model=settings.KNOWLEDGE_EMBED_MODEL)
    _ollama_ready = True
    print(f'Ollama embedder ready ({settings.KNOWLEDGE_EMBED_MODEL})')
except Exception as e:
    _ollama_ready = False
    print(f'Ollama not available: {e}')
    print('Search cells below will be skipped.')

In [ ]:
def search(query: str, n: int = 8, access_level: str | None = None):
    """Embed query and retrieve top-n chunks from ChromaDB."""
    if not _ollama_ready:
        print('Ollama not available')
        return

    where = {'access_level': {'$eq': access_level}} if access_level else None
    emb   = embed_fn.embed([query])

    res = col.query(
        query_embeddings=emb,
        n_results=n,
        where=where,
        include=['metadatas', 'documents', 'distances'],
    )

    for rank, (meta, doc, dist) in enumerate(
        zip(res['metadatas'][0], res['documents'][0], res['distances'][0]), 1
    ):
        print(f'--- rank {rank} | dist={dist:.4f} | chunk_index={meta.get("chunk_index")} ---')
        print(f'breadcrumb : {meta.get("breadcrumb")}')
        print(f'headline   : {meta.get("headline")}')
        print(f'topic      : {meta.get("topic")}')
        print(doc[:400])
        print()

In [ ]:
# Try a few queries
search('how does karma work in combat')

In [ ]:
search('what disciplines are available to humans')

In [ ]:
search('spell matrix and casting', access_level='player_visible')

## 6. Ad-hoc filtering

In [ ]:
# Filter by topic substring
TOPIC_FILTER = 'magic'
matches = df[df['topic'].str.contains(TOPIC_FILTER, case=False, na=False)]
print(f'{len(matches)} chunks matching topic~"{TOPIC_FILTER}"')
display(matches[['chunk_index','_ot_len','topic','breadcrumb','headline']].head(20))

In [ ]:
# Filter by text content
TEXT_FILTER = 'Versatility'
matches = df[df['original_text'].str.contains(TEXT_FILTER, case=False, na=False)]
print(f'{len(matches)} chunks containing "{TEXT_FILTER}"')
for _, row in matches.iterrows():
    show_chunk(row, chars=400)
    print('=' * 60)

In [ ]:
# Filter GM-only chunks by topic area
gm_only = df[df['access_level'] == 'gm_only']
print(f'GM-only chunks: {len(gm_only)}')
display(gm_only['topic'].value_counts().head(20))

## 7. Diagnostic summary

In [ ]:
# Print a consolidated health-check table
checks = {
    'stubs (<150 chars)':        (df['_ot_len'] < 150).sum(),
    'giants (>10k chars)':       (df['_ot_len'] > 10000).sum(),
    'encoding artifacts':        df.get('_has_artifact', df['original_text'].str.contains('\ufffd')).sum(),
    'image placeholders':        df['original_text'].str.contains('==> picture').sum(),
    'TOC dot-leaders':           df['original_text'].str.contains('\.{5,}').sum(),
    'stranded page numbers':     df['_has_pagenum'].sum() if '_has_pagenum' in df else '(run cell above)',
    'garbled OCR (ratio>0.35)':  (df['_garbled'] > 0.35).sum() if '_garbled' in df else '(run cell above)',
    'mid-word hyphen breaks':    df['_has_linebreak'].sum() if '_has_linebreak' in df else '(run cell above)',
    'bold/short breadcrumbs':    (df['breadcrumb'].str.contains(r'\*\*|_\*|\*_', regex=True) | (df['breadcrumb'].str.len() < 25)).sum(),
}

print(f'Collection: {COLLECTION} | Total: {len(df)} chunks\n')
for name, count in checks.items():
    pct = f'{count/len(df)*100:.1f}%' if isinstance(count, (int, float)) else ''
    print(f'  {name:<35} {count!s:>5}  {pct}')